# **NOTE**

##NOTE: Just put raw HTML and Text into "Init raw HTML & TEXT" cells

#Prepare dataset

In [ ]:
!pip install huggingface_hub
from huggingface_hub import login

hf_token=""   
login(hf_token)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="macroni2002/bom_cropped_objects",
    repo_type="dataset",
    local_dir="./cropped_objects",
    ignore_patterns=["README.md", ".gitattributes", ".jsonl", ".DS_Store", ".cache"],
)

In [ ]:
!pip install --upgrade transformers
!pip install -U accelerate
# !pip install git+https://github.com/huggingface/transformers.git

In [ ]:
import transformers
print(transformers.__version__)

#Init

## Install + import dependencies

In [ ]:
!pip install beautifulsoup4
!pip install sentence-transformers
!pip install rank-bm25
!pip install rapidfuzz
!pip install faiss-cpu
!pip install torch

In [ ]:
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

In [ ]:
import re
import json
import faiss
import unicodedata
import numpy as np
from bs4 import BeautifulSoup
from collections import defaultdict
from rapidfuzz import fuzz
from rank_bm25 import BM25Okapi
from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

##Init Model

###*Embedding model*

In [ ]:
embedding_model = SentenceTransformer(
    "BAAI/bge-m3"
)

cross_encoder = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

###*VLM model*

In [ ]:
# model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
model_name = "Qwen/Qwen3-VL-8B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_name,
)

vlm_model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
)


#Pre-process text func helper

In [ ]:
# =========================================================
# MERGE
# =========================================================
def merge_texts(texts):
    merged_text = "\n\n".join(texts)
    result = f"""
    {merged_text}
    """
    print(result)
    return result

# =========================================================
# STOPWORDS
# =========================================================
EN_STOPWORDS = {
    "the","and","of","to","in","on","for",
    "with","is","are","this","that"
}

VI_STOPWORDS = {
    "là","và","của","cho","trong","có","các",
    "được","với","ở","ra","vào","trên","dưới",
    "này","đó","thế"
}

VI_CHAR_PATTERN = re.compile(
    r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễ"
    r"ìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữỳýỵỷỹđ]"
)

# =========================================================
# NORMALIZE
# =========================================================
def normalize_text(text):
    text = text.lower().strip()
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[^\w\s\+\-\./]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_query(q):
    q = normalize_text(q)
    if len(q) <= 1:
        return ""

    if re.fullmatch(r"\d+", q):
        return ""
    return q

def clean_query_text(text):
    text = normalize_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# =========================================================
# LANGUAGE DETECT
# =========================================================
def is_vietnamese(text):
    vi_chars = len(VI_CHAR_PATTERN.findall(text))
    return vi_chars / max(len(text), 1) > 0.02

# =========================================================
# TOKENIZER
# =========================================================
def tokenize_en(text):
    text = normalize_text(text)
    tokens = text.split()
    return [t for t in tokens if len(t) > 1 and t not in EN_STOPWORDS]

def tokenize_vi(text):
    text = normalize_text(text)
    tokens = text.split()
    return [t for t in tokens if len(t) > 1 and t not in VI_STOPWORDS]

def hybrid_tokenize(text):
    if not text:
        return []

    if is_vietnamese(text):
        return tokenize_vi(text)

    return tokenize_en(text)

# =========================================================
# SENTENCE SPLIT + PHRASES
# =========================================================
def split_sentences(text):
    sentences = re.split(r"[.;:\n]", text)
    return [clean_query_text(x) for x in sentences if clean_query_text(x)]

def generate_ngrams(sentence, min_n=1, max_n=5):
    words = sentence.split()
    ngrams = []

    for n in range(min_n, max_n + 1):
        for i in range(len(words) - n + 1):
            gram = " ".join(words[i:i+n]).strip()
            if len(gram) <= 2:
                continue

            if re.fullmatch(r"\d+", gram):
                continue

            ngrams.append(gram)

    return ngrams

def extract_phrases(text, min_n=1, max_n=5):
    sentences = split_sentences(text)
    all_phrases = []

    for s in sentences:
        all_phrases.extend(generate_ngrams(s, min_n, max_n))

    return list(set(all_phrases))

#Define Hybrid Retriever

In [ ]:
# =========================================================
# HYBRID RETRIEVER
# =========================================================

class HybridRetriever:
    def __init__(
        self,
        corpus,
        embedding_model,
        cross_encoder,
        faiss_index,
        doc_embeddings
    ):

        self.corpus = corpus
        self.embed_model = embedding_model
        self.cross_encoder = cross_encoder
        self.index = faiss_index
        self.doc_embeddings = doc_embeddings
        self.query_cache = {}

        # -----------------------------
        # BM25 INIT
        # -----------------------------
        # BM25

        self.tokenized_corpus = [
            hybrid_tokenize(doc) for doc in corpus
        ]

        self.bm25 = BM25Okapi(self.tokenized_corpus)

    # =====================================================
    # NORMALIZE
    # =====================================================
    def _normalize(self, values):
        if not values:
            return values

        min_v, max_v = min(values), max(values)
        if max_v - min_v < 1e-6:
            return [0.0] * len(values)

        return [(v - min_v) / (max_v - min_v) for v in values]

    # =====================================================
    # Query embedding cache
    # =====================================================
    def _get_query_embedding(self, query):
        if query in self.query_cache:
            return self.query_cache[query]

        vec = self.embed_model.encode(
            [query],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        # limit cache

        if len(self.query_cache) > 10000:
            self.query_cache.clear()

        self.query_cache[query] = vec
        return vec

    # -------------------------------------------------
    # fuzzy retrieval
    # -------------------------------------------------
    def fuzzy_search(
        self,
        query,
        top_k=20
    ):
        results = []
        for i, doc in enumerate(self.corpus):
            score = fuzz.token_set_ratio(query, doc) / 100.0

            results.append({
                "id": i,
                "term": doc,
                "fuzzy_score": score
            })

        results = sorted(
            results,
            key=lambda x: x["fuzzy_score"],
            reverse=True
        )
        return results[:top_k]

    # -------------------------------------------------
    # embedding retrieval via FAISS
    # -------------------------------------------------
    def embedding_search(
        self,
        query,
        top_k=20
    ):
        q_vec = self._get_query_embedding(query)
        q_vec = q_vec.astype("float32")
        scores, ids = self.index.search(
            q_vec,
            top_k
        )

        results = []

        for score, idx in zip(
            scores[0],
            ids[0]
        ):
            results.append({
                "id": int(idx),
                "term": self.corpus[idx],
                "embed_score": float(score)
            })
        return results

    # =====================================================
    # BM25 SEARCH
    # =====================================================
    def bm25_search(
        self,
        query,
        top_k=20
    ):
        tokenized_query = hybrid_tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        results = []

        for idx, score in enumerate(scores):
            results.append({
                "id": idx,
                "term": self.corpus[idx],
                "bm25_score": float(score)
            })

        results = sorted(
            results,
            key=lambda x: x["bm25_score"],
            reverse=True
        )
        return results[:top_k]

    # -------------------------------------------------
    # merge candidates
    # -------------------------------------------------
    def merge_candidates(
        self,
        fuzzy_results,
        embed_results,
        bm25_results
    ):
        merged = {}
        # fuzzy
        for r in fuzzy_results:
            idx = r["id"]
            merged[idx] = {
                "id": idx,
                "term": r["term"],
                "fuzzy_score": r["fuzzy_score"],
                "embed_score": 0.0,
                "bm25_score": 0.0
            }

        # embedding
        for r in embed_results:
            idx = r["id"]
            if idx not in merged:
                merged[idx] = {
                    "id": idx,
                    "term": r["term"],
                    "fuzzy_score": 0.0,
                    "embed_score": r["embed_score"],
                    "bm25_score": 0.0
                }
            else:
                merged[idx]["embed_score"] = (
                    r["embed_score"]
                )

        # bm25
        for r in bm25_results:
            idx = r["id"]
            if idx not in merged:
                merged[idx] = {
                    "id": idx,
                    "term": r["term"],
                    "fuzzy_score": 0.0,
                    "embed_score": 0.0,
                    "bm25_score": r["bm25_score"]
                }
            else:
                merged[idx]["bm25_score"] = r["bm25_score"]
        return list(
            merged.values()
        )

    # -------------------------------------------------
    # cross encoder rerank
    # -------------------------------------------------
    def rerank(
        self,
        query,
        candidates,
        top_k=5
    ):
        if not candidates:
            return []

        # normalize
        fuzzy_vals = [c["fuzzy_score"] for c in candidates]
        embed_vals = [c["embed_score"] for c in candidates]
        bm25_vals = [c["bm25_score"] for c in candidates]
        fuzzy_norm = self._normalize(fuzzy_vals)
        embed_norm = self._normalize(embed_vals)
        bm25_norm = self._normalize(bm25_vals)

        pairs = [
            [query, c["term"]]
            for c in candidates
        ]

        ce_scores = self.cross_encoder.predict(
            pairs,
            batch_size=32
        )

        results = []
        for i, (c, ce_score) in enumerate(zip(candidates, ce_scores)):
            final_score = (
                0.20 * fuzzy_norm[i] +
                0.10 * bm25_norm[i] +
                0.30 * embed_norm[i] +
                0.40 * float(ce_score)
            )

            results.append({
                "term": c["term"],
                "score": final_score,
                "cross_score": float(ce_score),
                "fuzzy_score": c["fuzzy_score"],
                "embed_score": c["embed_score"],
                "bm25_score": c["bm25_score"]
            })

        results = sorted(
            results,
            key=lambda x: x["score"],
            reverse=True
        )

        return results[:top_k]

    # -------------------------------------------------
    # retrieve pipeline
    # -------------------------------------------------
    def retrieve(
        self,
        query,
        top_k=5
    ):
        query = clean_query(query)
        if not query:
            return []

        # fuzzy retrieval
        fuzzy_results = self.fuzzy_search(query, top_k=20)
        # embedding retrieval
        embed_results = self.embedding_search(query, top_k=20)
        # bm25 retrieval
        bm25_results = self.bm25_search(query, top_k=20)

        # merge
        candidates = self.merge_candidates(
            fuzzy_results,
            embed_results,
            bm25_results
        )

        # rerank
        final_results = self.rerank(
            query,
            candidates,
            top_k=top_k
        )
        return final_results

#Aggregator

In [ ]:
# =========================================================
# AGGREGATOR
# =========================================================
def aggregate_results(
    retriever,
    queries,
    top_k_each=5
):
    # total score across all OCR queries
    global_scores = defaultdict(float)

    # count how many times term appears
    hit_counts = defaultdict(int)

    # detailed mapping
    query_mapping = defaultdict(list)

    for q in queries:
        query = clean_query(q)

        if not query:
            continue

        results = retriever.retrieve(
            query,
            top_k=top_k_each
        )

        print(f"\n========== QUERY: {q} ==========\n")
        for r in results:
            term = r["term"]
            score = r["score"]

            print(
                f"{term:<25}"
                f" final={score:.4f}"
                f" | cross={r['cross_score']:.4f}"
                f" | fuzzy={r['fuzzy_score']:.4f}"
                f" | embed={r['embed_score']:.4f}"
                f" | bm25={r['bm25_score']:.4f}"
            )

            # accumulate
            global_scores[term] += score
            hit_counts[term] += 1

            query_mapping[q].append({
                "term": term,
                "score": score
            })

    # =====================================================
    # FINAL AGGREGATION
    # =====================================================
    final_results = []

    for term in global_scores:
        total_score = global_scores[term]
        hits = hit_counts[term]

        # boost repeated hits
        boosted_score = (
            total_score *
            (1 + 0.15 * hits)
        )

        final_results.append({
            "term": term,
            "total_score": total_score,
            "boosted_score": boosted_score,
            "hits": hits
        })

    final_results = sorted(
        final_results,
        key=lambda x: x["boosted_score"],
        reverse=True
    )

    return final_results, query_mapping

#Extract final term of corpus

In [ ]:
def final_terms(corpus, html_or_text, is_text=False):
    # =========================================================
    # BUILD CORPUS
    # =========================================================
    corpus = [
        normalize_text(x)
        for x in corpus.split(",")
        if x.strip()
    ]

    print("\n========== CORPUS ==========\n")

    print(corpus)

    # =========================================================
    # EMBEDDINGS
    # =========================================================

    doc_embeddings = embedding_model.encode(
        corpus,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    doc_embeddings = doc_embeddings.astype(
        "float32"
    )

    # =========================================================
    # FAISS INDEX
    # =========================================================

    dim = doc_embeddings.shape[1]

    index = faiss.IndexFlatIP(dim)

    index.add(doc_embeddings)

    print("\n========== FAISS ==========\n")
    print("Total vectors:", index.ntotal)

    # =========================================================
    # INIT RETRIEVER
    # =========================================================

    retriever = HybridRetriever(
        corpus=corpus,
        embedding_model=embedding_model,
        cross_encoder=cross_encoder,
        faiss_index=index,
        doc_embeddings=doc_embeddings
    )

    text_contents = []
    if is_text:
        cleaned_text = clean_query_text(
            html_or_text
        )

        print("\n========== CLEANED TEXT ==========\n")
        print(cleaned_text)

        # =========================================================
        # EXTRACT PHRASES
        # =========================================================
        text_contents = extract_phrases(
            cleaned_text,
            min_n=1,
            max_n=4
        )

        print("\n========== PHRASES ==========\n")
        for p in text_contents[:100]:
            print(p)
    else:
        soup = BeautifulSoup(
            html_or_text,
            "html.parser"
        )

        texts = list(
            soup.stripped_strings
        )

        print("\n========== RAW TABLE OCR TEXT ==========\n")
        print(texts)


        # =========================================================
        # CLEAN + DEDUP OCR TEXT
        # =========================================================

        text_contents = list(set(
            clean_query(x)
            for x in texts
            if clean_query(x)
        ))

        print("\n========== CLEANED OCR TEXT ==========\n")
        print(text_contents)




    # =========================================================
    # AGGREGATE ALL OCR TEXT
    # =========================================================

    final_results, query_mapping = aggregate_results(
        retriever=retriever,
        queries=text_contents,
        top_k_each=5
    )

    # =========================================================
    # RUN
    # =========================================================

    for text in text_contents:
        query = None
        if text:
            query = clean_query_text(text)
        else:
            query = clean_query(text)

        if not query:
            continue

        print(f"\n\n========== QUERY: {text} ==========\n")

        results = retriever.retrieve(
            query,
            top_k=5
        )

        for r in results:
            print(
                f"{r['term']:<25}"
                f" final={r['score']:.4f}"
                f" | cross={r['cross_score']:.4f}"
                f" | fuzzy={r['fuzzy_score']:.4f}"
                f" | embed={r['embed_score']:.4f}"
            )


    # =========================================================
    # FINAL RESULTS
    # =========================================================

    print("\n\n")
    print("=" * 60)
    print("FINAL AGGREGATED RESULTS")
    print("=" * 60)

    for r in final_results:
        print(
            f"{r['term']:<25}"
            f" boosted={r['boosted_score']:.4f}"
            f" | total={r['total_score']:.4f}"
            f" | hits={r['hits']}"
        )

    # =========================================================
    # FINAL TERM LIST
    # =========================================================

    final_terms = [
        r["term"]
        for r in final_results
    ]

    print("\n")
    print("=" * 60)
    print("FINAL TERM LIST")
    print("=" * 60)

    print(final_terms)
    return final_terms

#Init Prompt

In [ ]:
def vlm_prompt(terms, html_or_text, is_text=False):
    prompt = """"""
    if is_text:
        prompt = f"""
            You are correcting OCR text from a Vietnamese or English mechanical engineering document.
            TASK:
            Fix OCR errors using:

            1. Original image
            2. OCR raw text
            3. Engineering terminology candidates

            RULES:
            - Preserve sentence structure
            - Preserve numbering
            - Preserve dimensions
            - Preserve technical meaning
            - Correct accents
            - Prefer engineering terminology
            - Do NOT paraphrase
            - Do NOT summarize
            - Correct only when confident
            - Use image as ground truth

            TEXT:
            {html_or_text}
            ENGINEERING TERMS:
            {terms}
            Return ONLY corrected text.
            """
    else:
        prompt = f"""
            You are given:
            1. The original table image
            2. The OCR-extracted HTML

            Task: Fix OCR errors in the HTML table text.

            Requirements:
            - Use the image as the ground truth
            - Preserve the EXACT HTML structure (table, tr, td, colspan, rowspan)
            - DO NOT add, remove, or reorder any HTML tags
            - ONLY modify text inside HTML tags (text nodes)

            Text correction rules:
            - Correct OCR mistakes (spelling, accents, casing)
            - Preserve the original language (Vietnamese or English as in the image)
                - If text is ALL CAPS → keep ALL CAPS
                - If text is Title Case → keep Title Case
                - If text is lowercase → keep lowercase
            - Do NOT normalize casing unless it is clearly an OCR mistake
            - DO NOT change technical codes, numbers, dimensions, or part numbers unless clearly incorrect
            - Keep mechanical/engineering terminology accurate
            - Use the following terminology corpus as reference when applicable:
            {terms}

            HTML:
            {html_or_text}

            Return ONLY the corrected HTML.
            No explanations.
            """
    return prompt

#Query - Response

In [ ]:
def query_vlm(model, prompt, image_path):
    messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    img_pil = Image.open(image_path).convert("RGB")
    inputs = processor(
            text=[text],
            images=[img_pil],
            return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5120
    )

    generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

    response = processor.tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response

#Init Corpus

In [ ]:
# =========================================================
# CORPUS
# =========================================================

corpus = """
bản vẽ cơ khí, bản vẽ kỹ thuật, bản vẽ chi tiết, bản vẽ lắp, bản vẽ chế tạo, bản vẽ tổng thể, hình họa,
hình biểu diễn, hình chiếu, hình chiếu đứng, hình chiếu bằng, hình chiếu cạnh, hình chiếu trục đo, hình chiếu phối cảnh, hình chiếu phụ, hình chiếu khai triển,
hình chiếu thứ nhất, hình chiếu thứ ba, hình trích, hình chiếu phụ,
mặt cắt, mặt cắt toàn phần, mặt cắt cục bộ, mặt cắt một nửa, mặt cắt bậc, mặt cắt xoay, mặt cắt trích,
chi tiết máy, chốt chặn,

cỡ, kích cỡ, kích thước, đường kích thước, đường gióng, dung sai, dung sai kích thước, dung sai hình học,
sai lệch giới hạn, sai lệch trên, sai lệch dưới, kích thước danh nghĩa, kích thước thực, kích thước tham khảo, kích thước trong ngoặc,
ghi chú, chú thích, ký hiệu, ký hiệu dung sai, ký hiệu kỹ thuật,
khung bản vẽ, khung tên, tỷ lệ, đơn vị, số bản vẽ, số tờ, lần sửa đổi, bảng kê chi tiết, bảng vật liệu, danh mục chi tiết, người thiết kế, người kiểm tra, ngày vẽ,

độ phẳng, độ thẳng, độ tròn, độ trụ, độ song song, độ vuông góc, độ nghiêng, độ đồng tâm, độ đồng trục, độ đối xứng,
độ đảo, độ đảo hướng tâm, độ đảo mặt đầu, độ đảo toàn phần, dung sai vị trí, dung sai hình dạng, dung sai biên dạng,
vị trí thực, miền dung sai, mặt chuẩn, chuẩn định vị, chuẩn thiết kế, chuẩn công nghệ, mục tiêu chuẩn, hệ thống chuẩn,
điều kiện vật liệu tối đa, mmc, điều kiện vật liệu ít nhất, lmc,

lỗ, lỗ suốt, lỗ mù, lỗ bậc, lỗ ren, lỗ mồi, lỗ định vị, lỗ chìm,
ren, ren ngoài, ren trong, ren hệ mét, ren hệ inch, ren trái, ren phải, bước ren, số đầu mối ren,
rãnh, rãnh then, rãnh tròn, rãnh chữ t, rãnh thoát dao, rãnh thoát ren, rãnh thoát mài,
vát mép, bo góc, góc lượn, bán kính lượn, độ vát, độ dốc,
gân, gân tăng cứng, gờ, bậc, mặt phẳng, mặt trụ, mặt côn, gờ trục, bậc lỗ,

đường kính, đường kính ngoài, đường kính trong, đường kính danh nghĩa, phi,
bán kính, bán kính trong, bán kính ngoài,
chiều dày, chiều dày thành, chiều sâu, khoảng cách, chiều dài, chiều rộng, chiều cao, kích thước tổng thể, kích thước lắp,

độ nhám, độ nhám bề mặt, độ nhám ra, độ nhám rz, độ nhám ry, cấp độ nhám,
bề mặt gia công, bề mặt không gia công, ký hiệu độ nhám, hướng vân gia công,
vật liệu, thép, thép cacbon, thép hợp kim, nhôm, hợp kim nhôm, gang, đồng, nhựa, sắt, sắt tây,
ct3, c45, s45c, sus304, sus316, skd11, ss400,
xử lý nhiệt, tôi, ram, ủ, thường hóa, mạ, mạ kẽm, mạ crom, sơn phủ,
thấm cacbon, nitơ hóa, tôi cao tần, tôi phân cấp, thấm nitơ-cacbon, anode hóa, nhuộm đen, phốt phát hóa, phun cát,
độ cứng, hrc, hb, hv,

gia công, gia công cơ khí, tiện, phay, khoan, doa, taro, cắt, mài, hàn, đúc, rèn, dập, gia công cnc,
gia công thô, gia công tinh, lượng dư gia công, bavia, ba via, mép sắc, phá bavia, bo tròn cạnh, làm nhẵn,

bu lông, bu lông lục giác, bu lông đầu tròn, bu lông neo, bu lông chữ t, bu lông cường độ cao, bu lông inox,
đai ốc, đai ốc lục giác, đai ốc hãm, đai ốc cánh, đai ốc khóa, đai ốc mỏng, đai ốc tự hãm,
vít, vít tự ren, vít chìm, vít đầu trụ, vít đầu bằng, vít bake, vít lục giác chìm, vít gỗ,
vòng đệm, vòng đệm phẳng, vòng đệm vênh, vòng đệm lò xo, vòng đệm khóa, vòng đệm cao su, vòng đệm o-ring,
chốt, chốt trụ, chốt côn, chốt chẻ, chốt đàn hồi, chốt định vị, đinh tán, đinh tán đặc, đinh tán rỗng, đinh tán kéo,
phe cài, phe cài trục, phe cài lỗ, vòng chặn, vòng phớt,

then, then bằng, then bán nguyệt, then hoa, then dẫn, then trượt, then hoa trục, rãnh then,
ổ lăn, vòng bi, ổ bi đỡ, ổ bi chặn, ổ bi cầu, ổ bi đũa, ổ bi côn, ổ bi tì,
ổ trượt, bạc trượt, bạc lót, bạc đạn, bọc táp, bạc lót tự bôi trơn,
lò xo, lò xo nén, lò xo kéo, lò xo xoắn, lò xo đĩa,

bánh răng, bơm bánh răng, bánh răng trụ, bánh răng côn, bánh răng nghiêng, bánh răng thẳng, bánh răng trong, bánh răng ngoài,
thanh răng, trục vít, bánh vít, truyền động bánh răng, truyền động trục vít bánh vít,
trục, trục quay, trục dẫn, trục bị động, trục chính, trục then hoa, đầu trục, cổ trục,
khớp nối, khớp nối cứng, khớp nối mềm, khớp nối đàn hồi, khớp nối bánh răng, khớp nối trục, khớp các đăng, khớp nối vạn năng,
puly, dây đai, bánh đai, truyền động đai, xích, đĩa xích, truyền động xích, cam, cần gạt, tay quay,

van, van bi, van một chiều, van an toàn, van điều áp,
 ống dẫn, ống thép, ống nhựa, ống thủy lực, ống khí nén,
mặt bích, gioăng, gioăng cao su, phớt, phớt dầu, phớt chặn dầu, phớt cơ khí, phớt chắn bụi, khớp nối nhanh,

vỏ, nắp, thân máy, bệ máy, khung, kết cấu, tấm, tấm đế, tấm chắn, tấm chắn bụi, vỏ hộp, nắp đậy, nắp che,
lắp ghép, mối ghép, mối ghép ren, mối ghép then, mối ghép hàn, mối ghép bu lông,
dung sai lắp ghép, lắp ghép chặt, lắp ghép lỏng, lắp ghép trung gian, hệ thống lỗ chuẩn, hệ thống trục chuẩn,
khe hở, độ dôi, độ ép, lắp ghép có độ dôi, lắp ghép có khe hở, định vị, kẹp chặt,
ổ đỡ, gối đỡ, giá đỡ trục, hộp giảm tốc, bộ truyền, cụm chi tiết, cụm lắp, cụm máy,
lắp ráp, tháo lắp, trình tự lắp, vị trí lắp, sai số lắp ráp,
dầu bôi trơn, mỡ bôi trơn, bôi trơn, lỗ bôi trơn, rãnh bôi trơn, ống dẫn dầu, ống dẫn khí,

tải trọng, lực tác dụng, mô men, mô men xoắn, ứng suất, biến dạng, nén, kéo, xoắn, uốn,

⌀, phi, ø, bán kính, r, cộng trừ, ±, hình vuông, □, độ, °, phút, ', giây, ",
lỗ bậc, ⌴, lỗ chìm, ⌵, độ sâu, ↧, chiều dày, t, độ dốc, ∠, độ vát, ◅,
độ phẳng, ▱, độ thẳng, ⏤, độ tròn, ◯, độ trụ, ⌭, độ song song, ∥, độ vuông góc, ⊥,
độ đảo hướng tâm, ↗, độ đảo toàn phần, ⌯, độ đối xứng, ⌯, độ đồng tâm, ◎, vị trí thực, ⌖,
điều kiện vật liệu tối đa, Ⓜ, điều kiện vật liệu ít nhất, Ⓛ, bề mặt chuẩn, ▕,
đường tâm, cl, điểm chuẩn, cp, tham khảo, ref, nguyên mẫu, typ
"""

# Init raw HTML & TEXT


In [ ]:
# =========================================================
# 2. HTML OCR
# =========================================================

raw_html = """<table><tr><td>14</td><td colspan="2">Bac lot</td><td>3</td><td>Động tranh</td><td></td></tr><tr><td>13</td><td colspan="2">vòng đếm</td><td>1</td><td>Thếp CT3</td><td></td></tr><tr><td>12</td><td colspan="2">Chất trù 5nh55</td><td>2</td><td>Thếp CT3</td><td></td></tr><tr><td>11</td><td colspan="2">Vít M6 + 50</td><td>6</td><td>Thếp CT3</td><td></td></tr><tr><td>10</td><td colspan="2">Bu-lông M4 + 20</td><td>1</td><td>Thếp CT3</td><td></td></tr><tr><td>9</td><td colspan="2">Vùng đêm vườn</td><td>1</td><td>Thếp 65T</td><td></td></tr><tr><td>8</td><td colspan="2">Thên đông 4 + 4 + 14</td><td>1</td><td>Thếp CT3</td><td></td></tr><tr><td>7</td><td colspan="2">Chông chên</td><td>1</td><td>Thếp CT3</td><td></td></tr><tr><td>6</td><td colspan="2">Chất chên</td><td>1</td><td>Sựi đay</td><td></td></tr><tr><td>5</td><td colspan="2">Bac lét</td><td>1</td><td>Động tranh</td><td></td></tr><tr><td>4</td><td colspan="2">Giá đết</td><td>1</td><td>Gông 15-32</td><td></td></tr><tr><td>3</td><td colspan="2">Bình ràng</td><td>2</td><td>Thếp 45</td><td>mất 32+12</td></tr><tr><td>2</td><td colspan="2">Hợp bánh ràng</td><td>1</td><td>Gông 15-32</td><td></td></tr><tr><td>1</td><td colspan="2">Nếp</td><td>1</td><td>Gông 15-32</td><td></td></tr><tr><td>Với</td><td colspan="2">Tên cải tiết mấy</td><td>Sựi đự</td><td>Vật liệu</td><td>Ghi chú</td></tr><tr><td>Bản gỗ</td><td>L.X</td><td>cũ</td><td rowspan="2" colspan="3">BỘM BẢNH RĂNG</td></tr><tr><td>Can chinh</td><td>Tổng</td><td>1.99</td></tr><tr><td rowspan="2" colspan="3">Bộ môn Hình hào VKT Đại học Bách Hòa Hán Dưới</td><td rowspan="2" colspan="2">ĐẠN VỀ LẠP SỐ 3</td><td>Tý lệ</td></tr><tr><td>1:1</td></tr></table>"""
# raw_html = """<table><tr><td>17</td><td>True $ \alpha $ln</td><td>1</td><td>True $ \beta $45</td><td></td></tr><tr><td>16</td><td>Doi $ \beta $ n6p</td><td>1</td><td>True $ \gamma $CT3</td><td></td></tr><tr><td>15</td><td>True $ \gamma $ $ \alpha $ln</td><td>1</td><td>True $ \gamma $45</td><td></td></tr></table>"""

In [ ]:
# =========================================================
# 2. TEXT OCR
# =========================================================

raw_texts = [
        "Thuyết mình : Thần băm gồm các chi tiết máy chính là\ngà do 4, hợp đánh răng 2 và nấp 1, chúng được ghép\nkhỉ với nhau bằng hai chất dính vị 12 và sâu viết 11.",
        "Trong hợp bình răng 2 có hai trục 15 và 17, trên do\nlên chặt hai bình răng 3 : bình răng chù động ở trên và\nbình răng bị động ở dưới ; cặp bình răng này quay nhỏ\nchuyến động của bình răng bên ngoài (về bảng nét hai\nchấm gạch mạnh, lấp ở chỗ then 8).",
        "Các bánh răng 3 quay nhanh theo chiều mối tân sổ tạo ra sức hút từ 15 phía sau bám đổ kéo chất lỏng chạy vào các kế răng ; lên đổ, chất lỏng chuyển theo các kế răng này qua lỗ ra phía truổ. Cứ thế, chất lỏng dược hút và dậy lên tục qua bám vội đổ lụo lộn.",
        "Các bạc lốt 5 và 14 là các 6 truyện ở đâu hai trục -\nCác chi tiết 6, 7 dùng để chén khít không cho chết lòng\ncó ni ra ngoài."
      ]

# raw_texts = [
#   "Cả: ch: đất mày\nđể về tịch :\n! 2. 2. 4. 5.\n19. 6. 17"
# ]

In [ ]:
# =========================================================
# 2. IMAGE PATH
# =========================================================
image_path_table = "/content/cropped_objects/images/table_0.png"
image_path_note = "/content/cropped_objects/images/note_5.png"


# **MAIN**

##**BY VLM**

###**PREDICT 1**

####**HTML**

In [ ]:
terms_html_1 = final_terms(corpus=corpus, html_or_text=raw_html, is_text=False)
prompt_html_1 = vlm_prompt(terms=terms_html_1, html_or_text=raw_html, is_text=False)

In [ ]:
now = time.time()

html_predict_1 = query_vlm(model=vlm_model, prompt=prompt_html_1, image_path=image_path_table)
print(html_predict_1)

print(time.time() - now)

####**TEXT**

In [ ]:
# merge notes array to one text string
raw_text = merge_texts(raw_texts)

In [ ]:
terms_text_1 = final_terms(corpus=corpus, html_or_text=raw_text, is_text=True)
prompt_text_1 = vlm_prompt(terms=terms_text_1, html_or_text=raw_text, is_text=True)

In [ ]:
now = time.time()

text_predict_1 = query_vlm(model=vlm_model, prompt=prompt_text_1, image_path=image_path_note)
print(text_predict_1)

print(time.time() - now)

###**PREDICT 2**

####**HTML**

In [ ]:
terms_html_2 = final_terms(corpus=corpus, html_or_text=html_predict_1, is_text=False)
prompt_html_2 = vlm_prompt(terms=terms_html_2, html_or_text=html_predict_1, is_text=False)

In [ ]:
now = time.time()

html_predict_2 = query_vlm(model=vlm_model, prompt=prompt_html_2, image_path=image_path_table)
print(html_predict_2)

print(time.time() - now)

####**TEXT**

In [ ]:
terms_text_2 = final_terms(corpus=corpus, html_or_text=text_predict_1, is_text=True)
prompt_text_2 = vlm_prompt(terms=terms_text_2, html_or_text=text_predict_1, is_text=True)

In [ ]:
now = time.time()

text_predict_2 = query_vlm(model=vlm_model, prompt=prompt_text_2, image_path=image_path_note)
print(text_predict_2)

print(time.time() - now)